# Bitcoin Price Prediction using LSTM Neural Networks

This cleaned portfolio notebook reuses the packaged model and modular source code. It demonstrates
data preparation, feature engineering, sequence construction, one-step replay evaluation, baseline
comparison, and recursive multi-day forecasting.

> **Financial disclaimer:** This project is for educational and portfolio demonstration only. It is
> not financial advice, and its outputs must not be used for investment or trading decisions.


## Methodology improvements

The supplied notebook was preserved under `archive/original-project-files/`. The portfolio pipeline
documents and corrects two important issues for future retraining:

1. fit preprocessing only on the training period;
2. use separate chronological training, validation, and untouched test periods.

The included packaged model remains the supplied trained artifact. The cleaned retraining code is in
`src/model_training.py`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.cloud_inference import NumpyBitcoinLSTM
from src.config import CONFIG_PATH, SAMPLE_DATA_PATH, SCALER_PATH, WEIGHTS_PATH
from src.data_preprocessing import clean_market_data
from src.feature_engineering import FEATURE_COLUMNS, create_market_features
from src.forecasting_pipeline import forecast_future, load_json, load_scaler, replay_predictions
from src.model_evaluation import baseline_predictions, regression_metrics


## 1. Load and validate the packaged offline dataset

In [ ]:
market = clean_market_data(pd.read_csv(SAMPLE_DATA_PATH))
market.info()
market.head()


The packaged CSV is deterministic synthetic OHLCV demonstration data. It keeps the
application and notebook functional without depending on internet access. The original notebook
used Yahoo Finance `BTC-USD` history from 2018 through 2024.

In [ ]:
market.plot(x="Date", y="Close", figsize=(12, 5), title="Bitcoin Closing-Price Demonstration Series")
plt.ylabel("Close (USD)")
plt.show()


## 2. Create the exact five model features

In [ ]:
features = create_market_features(market)
print(FEATURE_COLUMNS)
features[["Date", *FEATURE_COLUMNS]].head()


In [ ]:
features.plot(x="Date", y=["Close", "SMA_7", "SMA_30"], figsize=(12, 5))
plt.title("Close and Moving Averages")
plt.ylabel("USD")
plt.show()


## 3. Load backend-free pretrained inference artifacts

In [ ]:
model = NumpyBitcoinLSTM(WEIGHTS_PATH)
scaler = load_scaler(SCALER_PATH)
config = load_json(CONFIG_PATH)
look_back = int(config["look_back"])
print(f"Input window: {look_back} days × {len(FEATURE_COLUMNS)} features")


## 4. Generate one-step replay predictions

In [ ]:
replay = replay_predictions(market, model, scaler, look_back)
test_replay = replay.tail(max(60, int(len(replay) * 0.20))).reset_index(drop=True)
metrics = regression_metrics(test_replay["Actual_Close"], test_replay["Predicted_Close"])
metrics


In [ ]:
ax = test_replay.plot(
    x="Date",
    y=["Actual_Close", "Predicted_Close"],
    figsize=(12, 5),
    title="One-Step Replay: Actual vs Predicted Close",
)
ax.set_ylabel("USD")
plt.show()


## 5. Compare transparent baselines

In [ ]:
baseline_frame = baseline_predictions(features["Close"]).dropna().reset_index(drop=True)
baseline_results = {
    column: regression_metrics(baseline_frame["Actual"], baseline_frame[column])
    for column in ["Naive", "Moving_Average_7", "Linear_Trend_30"]
}
pd.DataFrame(baseline_results).T


A naive previous-close forecast can outperform more complex price-level models because
adjacent daily closing prices are highly persistent. Baseline comparison is therefore essential.
High R² alone does not demonstrate a profitable or reliable trading signal.

## 6. Generate a recursive multi-day forecast

In [ ]:
future = forecast_future(market, model, scaler, look_back, horizon=30)
future.head()


In [ ]:
history_tail = market.tail(180)[["Date", "Close"]].rename(columns={"Close": "Historical_Close"})
ax = history_tail.plot(x="Date", y="Historical_Close", figsize=(12, 5))
future.plot(x="Date", y="Predicted_Close", ax=ax)
plt.title("Historical Close and 30-Day Recursive Forecast")
plt.ylabel("USD")
plt.show()


## Interpretation and limitations

- Recursive forecasts reuse earlier predictions, so uncertainty accumulates with horizon.
- The supplied artifact was created with preprocessing and validation limitations documented in
  `IMPROVEMENTS.md`.
- Historical OHLCV variables omit macroeconomic, regulatory, sentiment, on-chain, order-book, and
  derivatives information.
- This output is an ML workflow demonstration, not a recommendation or expected investment outcome.
